# 🚗 Car Purchase Amount Prediction — Deep Learning (ANN)

**Objective:** Build an end-to-end deep learning pipeline to predict how much a customer will spend on a car purchase based on their demographic and financial features.

**Dataset:** Car Purchasing Dataset (500 records, 9 columns)

**Model:** Artificial Neural Network (ANN) — Regression

---

### Notebook Structure
1. Imports & Setup
2. Load & Explore Data (EDA)
3. Data Preprocessing & Feature Engineering
4. Baseline ANN Model (No Regularization)
5. Improved ANN Model (L2 + Dropout + Early Stopping)
6. Hyperparameter Tuning (Keras Tuner)
7. Final Evaluation (Metrics & Plots)
8. Model Saving & Loading
9. Inference Pipeline
10. Conclusion & Limitations

---
## 1. Imports & Setup

In [ ]:
# Install Keras Tuner if not already installed
!pip install keras-tuner -q

In [ ]:
# Core
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

# Keras Tuner
import keras_tuner as kt

# Save/Load utilities
import joblib
import json

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f'TensorFlow version: {tf.__version__}')
print(f'Keras Tuner version: {kt.__version__}')

---
## 2. Load & Explore Data (EDA)

In [ ]:
# Load dataset from GitHub
url = 'https://raw.githubusercontent.com/Nawaf-Alorabi/Tw_Customer_Churn_Prediction_System_DeepLearn/8b7a5019a30f36a99d705ace4e3c94904509369f/car_purchasing.csv'
df = pd.read_csv(url, encoding='latin-1')

print(f'Dataset Shape: {df.shape}')
df.head()

In [ ]:
# Basic info
df.info()

In [ ]:
# Statistical summary
df.describe().round(2)

In [ ]:
# Check for missing values
print('Missing Values:')
print(df.isnull().sum())
print(f'\nDuplicate rows: {df.duplicated().sum()}')

In [ ]:
# Target variable distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['car purchase amount'], bins=30, color='#2196F3', edgecolor='white', alpha=0.85)
axes[0].set_title('Distribution of Car Purchase Amount', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Car Purchase Amount ($)')
axes[0].set_ylabel('Frequency')

axes[1].boxplot(df['car purchase amount'], vert=True)
axes[1].set_title('Box Plot — Car Purchase Amount', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Car Purchase Amount ($)')

plt.tight_layout()
plt.show()

In [ ]:
# Numeric features distribution
numeric_cols = ['gender', 'age', 'annual Salary', 'credit card debt', 'net worth']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].hist(df[col], bins=25, color='#4CAF50', edgecolor='white', alpha=0.85)
    axes[i].set_title(col, fontsize=12, fontweight='bold')
    axes[i].set_ylabel('Frequency')

axes[-1].axis('off')  # hide the 6th subplot
plt.suptitle('Feature Distributions', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
corr_cols = ['gender', 'age', 'annual Salary', 'credit card debt', 'net worth', 'car purchase amount']
corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Correlation Heatmap', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots: each feature vs target
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].scatter(df[col], df['car purchase amount'], alpha=0.5, s=20, color='#FF5722')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Car Purchase Amount')
    axes[i].set_title(f'{col} vs Target', fontsize=12, fontweight='bold')

axes[-1].axis('off')
plt.suptitle('Feature vs Car Purchase Amount', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Gender split on purchase amount
plt.figure(figsize=(8, 5))
df.groupby('gender')['car purchase amount'].mean().plot(kind='bar', color=['#2196F3', '#E91E63'], edgecolor='white')
plt.title('Average Car Purchase Amount by Gender', fontsize=14, fontweight='bold')
plt.xlabel('Gender (0 = Male, 1 = Female)')
plt.ylabel('Average Purchase Amount ($)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### EDA Key Findings
- **No missing values** — clean dataset.
- Target (`car purchase amount`) is roughly **normally distributed** (range ~$9K–$80K).
- **Strong positive correlations** with `age`, `annual Salary`, and `net worth`.
- `customer name`, `customer e-mail`, and `country` are **non-predictive** — will be dropped.
- `gender` is already binary encoded (0/1).

---
## 3. Data Preprocessing & Feature Engineering

In [ ]:
# Drop non-predictive columns
df_clean = df.drop(columns=['customer name', 'customer e-mail', 'country'])
print(f'Cleaned shape: {df_clean.shape}')
df_clean.head()

In [ ]:
# Define features and target
X = df_clean.drop(columns=['car purchase amount'])
y = df_clean['car purchase amount']

print(f'Features shape: {X.shape}')
print(f'Target shape:   {y.shape}')
print(f'\nFeature columns: {X.columns.tolist()}')

In [ ]:
# Train / Test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'X_train: {X_train.shape}  |  X_test: {X_test.shape}')
print(f'y_train: {y_train.shape}  |  y_test: {y_test.shape}')

In [ ]:
# Feature Scaling (StandardScaler)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Scaling applied. Sample of scaled training data:')
print(pd.DataFrame(X_train_scaled, columns=X.columns).head())

---
## 4. Baseline ANN Model (No Regularization)

In [ ]:
# Build baseline model — no regularization
baseline_model = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(1, activation='linear')  # Regression output
])

baseline_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

baseline_model.summary()

In [ ]:
# Train baseline model
baseline_history = baseline_model.fit(
    X_train_scaled, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

In [ ]:
# Evaluate baseline on test set
baseline_pred = baseline_model.predict(X_test_scaled).flatten()

print('=== Baseline Model Results ===')
print(f'MAE:  ${mean_absolute_error(y_test, baseline_pred):,.2f}')
print(f'RMSE: ${np.sqrt(mean_squared_error(y_test, baseline_pred)):,.2f}')
print(f'R²:   {r2_score(y_test, baseline_pred):.4f}')

In [ ]:
# Plot baseline training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(baseline_history.history['loss'], label='Train Loss', linewidth=2)
axes[0].plot(baseline_history.history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_title('Baseline — Loss (MSE)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE')
axes[0].legend()

axes[1].plot(baseline_history.history['mae'], label='Train MAE', linewidth=2)
axes[1].plot(baseline_history.history['val_mae'], label='Val MAE', linewidth=2)
axes[1].set_title('Baseline — MAE', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 5. Improved ANN Model (L2 + Dropout + Early Stopping)

**Regularization techniques applied:**
- **L2 Regularization** (`kernel_regularizer=l2`) — penalizes large weights
- **Dropout** — randomly drops neurons during training to reduce overfitting
- **Early Stopping** — monitors `val_loss` and stops training when it stops improving

In [ ]:
# Build improved model with regularization
improved_model = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.3),
    Dense(32, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.3),
    Dense(16, activation='relu', kernel_regularizer=l2(0.001)),
    Dense(1, activation='linear')
])

improved_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

improved_model.summary()

In [ ]:
# Callbacks
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

# Train improved model
improved_history = improved_model.fit(
    X_train_scaled, y_train,
    epochs=200,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# Evaluate improved model on test set
improved_pred = improved_model.predict(X_test_scaled).flatten()

print('=== Improved Model Results ===')
print(f'MAE:  ${mean_absolute_error(y_test, improved_pred):,.2f}')
print(f'RMSE: ${np.sqrt(mean_squared_error(y_test, improved_pred)):,.2f}')
print(f'R²:   {r2_score(y_test, improved_pred):.4f}')

In [ ]:
# Plot improved training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(improved_history.history['loss'], label='Train Loss', linewidth=2)
axes[0].plot(improved_history.history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_title('Improved — Loss (MSE)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE')
axes[0].legend()

axes[1].plot(improved_history.history['mae'], label='Train MAE', linewidth=2)
axes[1].plot(improved_history.history['val_mae'], label='Val MAE', linewidth=2)
axes[1].set_title('Improved — MAE', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'\nEarly Stopping triggered at epoch: {len(improved_history.history["loss"])}')

---
## 6. Hyperparameter Tuning (Keras Tuner)

In [ ]:
def build_tunable_model(hp):
    """
    Build a tunable ANN model for Keras Tuner.
    Tunes: number of layers, units per layer, dropout rate, learning rate, L2 strength.
    """
    model = Sequential()
    model.add(Input(shape=(X_train_scaled.shape[1],)))

    # Tune number of hidden layers (1 to 4)
    num_layers = hp.Int('num_layers', min_value=1, max_value=4, step=1)

    # Tune L2 regularization strength
    l2_strength = hp.Choice('l2_strength', values=[1e-2, 1e-3, 1e-4])

    for i in range(num_layers):
        units = hp.Int(f'units_{i}', min_value=16, max_value=128, step=16)
        model.add(Dense(units, activation='relu', kernel_regularizer=l2(l2_strength)))

        # Tune dropout rate
        dropout_rate = hp.Float(f'dropout_{i}', min_value=0.1, max_value=0.5, step=0.1)
        model.add(Dropout(dropout_rate))

    # Output layer
    model.add(Dense(1, activation='linear'))

    # Tune learning rate
    learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='mse',
        metrics=['mae']
    )

    return model

In [ ]:
# Initialize Keras Tuner (Bayesian Optimization)
tuner = kt.BayesianOptimization(
    build_tunable_model,
    objective='val_mae',
    max_trials=20,
    seed=42,
    directory='tuner_results',
    project_name='car_purchase_ann'
)

# Early stopping for tuner
tuner_early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

print('Starting hyperparameter search...')
tuner.search(
    X_train_scaled, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[tuner_early_stop],
    verbose=1
)

print('\nSearch complete!')

In [ ]:
# Show best hyperparameters
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]

print('=== Best Hyperparameters ===')
print(f'Number of layers: {best_hp.get("num_layers")}')
print(f'L2 strength:      {best_hp.get("l2_strength")}')
print(f'Learning rate:    {best_hp.get("learning_rate")}')

for i in range(best_hp.get('num_layers')):
    print(f'  Layer {i+1}: units={best_hp.get(f"units_{i}")}, dropout={best_hp.get(f"dropout_{i}")}')

In [ ]:
# Build and train the best model
best_model = tuner.hypermodel.build(best_hp)
best_model.summary()

In [ ]:
# Train the best model with full training data
best_early_stop = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

best_history = best_model.fit(
    X_train_scaled, y_train,
    epochs=200,
    batch_size=32,
    validation_split=0.2,
    callbacks=[best_early_stop],
    verbose=1
)

In [ ]:
# Evaluate tuned model on test set
best_pred = best_model.predict(X_test_scaled).flatten()

print('=== Tuned Model Results ===')
print(f'MAE:  ${mean_absolute_error(y_test, best_pred):,.2f}')
print(f'RMSE: ${np.sqrt(mean_squared_error(y_test, best_pred)):,.2f}')
print(f'R²:   {r2_score(y_test, best_pred):.4f}')

---
## 7. Final Evaluation (Metrics & Plots)

In [ ]:
# Compare all 3 models
models_results = {
    'Baseline (No Reg)': baseline_pred,
    'Improved (L2+Dropout+ES)': improved_pred,
    'Tuned (Keras Tuner)': best_pred
}

comparison = []
for name, preds in models_results.items():
    comparison.append({
        'Model': name,
        'MAE ($)': f'{mean_absolute_error(y_test, preds):,.2f}',
        'RMSE ($)': f'{np.sqrt(mean_squared_error(y_test, preds)):,.2f}',
        'R² Score': f'{r2_score(y_test, preds):.4f}'
    })

comparison_df = pd.DataFrame(comparison)
print('\n=== Model Comparison ===')
comparison_df

In [ ]:
# Final model training curves (best/tuned)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(best_history.history['loss'], label='Train Loss', linewidth=2)
axes[0].plot(best_history.history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_title('Tuned Model — Loss (MSE)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE')
axes[0].legend()

axes[1].plot(best_history.history['mae'], label='Train MAE', linewidth=2)
axes[1].plot(best_history.history['val_mae'], label='Val MAE', linewidth=2)
axes[1].set_title('Tuned Model — MAE', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Actual vs Predicted scatter plot (best model)
plt.figure(figsize=(8, 8))
plt.scatter(y_test, best_pred, alpha=0.6, s=40, color='#2196F3', edgecolors='white', linewidth=0.5)

# Perfect prediction line
min_val = min(y_test.min(), best_pred.min())
max_val = max(y_test.max(), best_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')

plt.xlabel('Actual Car Purchase Amount ($)', fontsize=12)
plt.ylabel('Predicted Car Purchase Amount ($)', fontsize=12)
plt.title('Actual vs Predicted — Tuned Model', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Residuals analysis
residuals = y_test.values - best_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residuals distribution
axes[0].hist(residuals, bins=25, color='#9C27B0', edgecolor='white', alpha=0.85)
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[0].set_title('Residuals Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Residual ($)')
axes[0].set_ylabel('Frequency')

# Residuals vs Predicted
axes[1].scatter(best_pred, residuals, alpha=0.5, s=30, color='#FF9800')
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[1].set_title('Residuals vs Predicted', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Predicted ($)')
axes[1].set_ylabel('Residual ($)')

plt.tight_layout()
plt.show()

---
## 8. Model Saving & Loading

In [ ]:
# Save the best model
best_model.save('car_purchase_model.h5')
print('Model saved: car_purchase_model.h5')

# Save the scaler
joblib.dump(scaler, 'scaler.pkl')
print('Scaler saved: scaler.pkl')

# Save feature names for inference
feature_names = X.columns.tolist()
with open('feature_names.json', 'w') as f:
    json.dump(feature_names, f)
print(f'Feature names saved: {feature_names}')

In [ ]:
# Test loading the model
loaded_model = load_model('car_purchase_model.h5')
loaded_scaler = joblib.load('scaler.pkl')

# Verify it works
test_pred = loaded_model.predict(loaded_scaler.transform(X_test)).flatten()
print(f'Loaded model R²: {r2_score(y_test, test_pred):.4f}')
print('Model loaded and verified successfully!')

---
## 9. Inference Pipeline

In [ ]:
def predict_car_purchase(customer_data: dict) -> dict:
    """
    Predict car purchase amount for a new customer.

    Parameters:
    -----------
    customer_data : dict
        Must contain: 'gender', 'age', 'annual Salary', 'credit card debt', 'net worth'

    Returns:
    --------
    dict with predicted purchase amount and input summary.
    """
    # Load model and scaler
    model = load_model('car_purchase_model.h5', compile=False)
    scaler = joblib.load('scaler.pkl')

    with open('feature_names.json', 'r') as f:
        feature_names = json.load(f)

    # Validate input
    for col in feature_names:
        if col not in customer_data:
            raise ValueError(f'Missing feature: {col}')

    # Create DataFrame from input
    input_df = pd.DataFrame([customer_data])[feature_names]

    # Scale
    input_scaled = scaler.transform(input_df)

    # Predict
    prediction = model.predict(input_scaled, verbose=0).flatten()[0]

    return {
        'predicted_car_purchase_amount': round(float(prediction), 2),
        'input': customer_data
    }

In [ ]:
# Test the inference pipeline
sample_customer = {
    'gender': 1,
    'age': 45,
    'annual Salary': 65000,
    'credit card debt': 12000,
    'net worth': 450000
}

result = predict_car_purchase(sample_customer)

print('=== Inference Result ===')
print(f'Customer Info:')
for k, v in result['input'].items():
    print(f'  {k}: {v}')
print(f'\nPredicted Car Purchase Amount: ${result["predicted_car_purchase_amount"]:,.2f}')

In [ ]:
# Test with another customer
sample_customer_2 = {
    'gender': 0,
    'age': 30,
    'annual Salary': 40000,
    'credit card debt': 5000,
    'net worth': 150000
}

result_2 = predict_car_purchase(sample_customer_2)
print(f'Customer 2 — Predicted Purchase: ${result_2["predicted_car_purchase_amount"]:,.2f}')

### (Optional) FastAPI Inference Script

Save the cell below as `app.py` and run with `uvicorn app:app --reload`

In [ ]:
# FastAPI script (save as app.py for deployment)
fastapi_code = '''
from fastapi import FastAPI
from pydantic import BaseModel
import numpy as np
import pandas as pd
import joblib
import json
from tensorflow.keras.models import load_model

app = FastAPI(title="Car Purchase Prediction API")

# Load once at startup
model = load_model("car_purchase_model.h5", compile=False)
scaler = joblib.load("scaler.pkl")
with open("feature_names.json", "r") as f:
    feature_names = json.load(f)

class CustomerInput(BaseModel):
    gender: int
    age: float
    annual_salary: float
    credit_card_debt: float
    net_worth: float

@app.post("/predict")
def predict(customer: CustomerInput):
    data = {
        "gender": customer.gender,
        "age": customer.age,
        "annual Salary": customer.annual_salary,
        "credit card debt": customer.credit_card_debt,
        "net worth": customer.net_worth
    }
    input_df = pd.DataFrame([data])[feature_names]
    input_scaled = scaler.transform(input_df)
    prediction = model.predict(input_scaled, verbose=0).flatten()[0]
    return {"predicted_car_purchase_amount": round(float(prediction), 2)}

@app.get("/")
def root():
    return {"message": "Car Purchase Prediction API is running!"}
'''

with open('app.py', 'w') as f:
    f.write(fastapi_code)

print('FastAPI script saved as app.py')
print('Run with: uvicorn app:app --reload')

---
## 10. Conclusion & Limitations

### Summary
- Built an **end-to-end ANN regression pipeline** to predict car purchase amounts.
- Applied **3 regularization techniques**: L2, Dropout, and Early Stopping.
- Used **Keras Tuner (Bayesian Optimization)** to find optimal hyperparameters.
- Saved model (`.h5`), scaler (`.pkl`), and feature names for reproducible inference.
- Created both a **Python inference function** and a **FastAPI deployment script**.

### Limitations
- **Small dataset** (500 records) — limits generalization.
- **No feature engineering** beyond scaling — could explore polynomial features or interaction terms.
- **No cross-validation** — single train/test split; k-fold would give more robust estimates.
- **`country` column dropped** — could be useful if one-hot encoded but adds ~200 columns.
- **No external validation** — testing only on held-out data from the same distribution.